# Synthetic NIR for 3-band NAIP and a 2005 canopy height model (Santa Clara)

The canopy height model requires four bands (R, G, B, NIR). The 2005 NAIP image for this quad has three. This notebook tests whether NIR predicted from R, G and B is good enough to run the model on the 2005 image.

The test uses the nine 4-band years that already have canopy height models on Drive. Each year is withheld in turn. A regressor trained on the other years predicts NIR for the withheld year from its RGB bands, the predicted NIR is compared with the real NIR, and the canopy height model is run on the synthetic image so that its output can be compared with the canopy height model made from the real NIR. The final section trains on all 4-band years, and separately on the 1 m years only, to produce two 2005 canopy height models.

Runtime: **Runtime > Change runtime type > T4 GPU**. Each canopy height inference takes about 150 s on a T4.

A cold run of the whole notebook takes roughly 2.5 to 3.5 hours. Every expensive product (pixel samples, fitted models, synthetic images, canopy height models, metric tables) is cached on Drive and skipped on rerun, so a disconnected session resumes where it stopped.

Outputs go to `MyDrive/SantaClara/nir_synthesis/`.

## 1. Configuration

`FEATURE_SETS` lists the feature sets evaluated at the pixel level. `FEATURE_SET_FOR_CHM` selects the one used for full-image prediction and canopy height inference. Change it to `"rgb_texture"` if the leave-one-out metrics show that texture matters. `TRAIN_SETS` defines the two training strategies for the final 2005 model: every 4-band year, or the 1 m years only.

`MODEL_KIND` is `"hgb"` (histogram gradient boosting) by default. `"rf"` switches to a random forest with `RF_PARAMS`.

In [ ]:
from pathlib import Path

import numpy as np

SITE = "SantaClara"
DRIVE_FOLDER = "SantaClara"

# Canopy height model (same settings as the main Santa Clara notebook)
NAIPCHM_REPO = Path("/content/naip-chm")
MODEL_CHECKPOINT = "model/model_20251016.pt"
MODEL_CONFIG = "configs/config.yaml"
CHIP_SIZE = 432
CHIP_OVERLAP = 0.2

# Target image (3 bands) and sampling
TARGET_YEAR = 2005
SEED = 0
N_SAMPLES = 100_000          # random valid pixels sampled per year
COLLAR_OPENING = False       # morphological opening of the valid mask (off)

# NIR regression
FEATURE_SETS = ("rgb", "rgb_texture")   # evaluated in the leave-one-out loop
FEATURE_SET_FOR_CHM = "rgb"             # used for full-image prediction and inference
TEXTURE_WINDOW = 5                      # pixels, native resolution
MODEL_KIND = "hgb"                      # "hgb" or "rf"
HGB_PARAMS = dict(
    max_iter=300,
    learning_rate=0.1,
    max_leaf_nodes=63,
    min_samples_leaf=50,
    early_stopping=False,
    random_state=SEED,
)
RF_PARAMS = dict(
    n_estimators=100,
    min_samples_leaf=5,
    max_samples=0.5,
    n_jobs=-1,
    random_state=SEED,
)
EARLY_YEARS = (2009, 2010, 2012, 2014)
TRAIN_SETS = {"all": None, "early": EARLY_YEARS}   # None means every 4-band year

# Error stratification by canopy height (cm, from the real-NIR CHM)
HEIGHT_BINS_CM = [0, 200, 500, 1000, 2000, np.inf]
HEIGHT_BIN_LABELS = ["0-2 m", "2-5 m", "5-10 m", "10-20 m", ">20 m"]

# Full-image prediction and output
BLOCK_ROWS = 512
PREDICT_CHUNK = 1_000_000
WRITE_MASK_BAND = True       # band 5 = validity mask, read by the model
KEEP_SYNNIR_IMAGES = True    # copy the 5-band synthetic images to Drive (about 0.3 GB each)
CHM_NODATA = 65535
DECIMATE = 6                 # map panels are decimated by this factor
SCATTER_N = 200_000          # sampled pixel pairs in scatter panels

RUN_TAG = f"{MODEL_KIND}_n{N_SAMPLES}_seed{SEED}"
print(f"{SITE}: target {TARGET_YEAR}, run tag {RUN_TAG}, CHM feature set {FEATURE_SET_FOR_CHM}")

## 2. Check the runtime

Fail here rather than an hour into a CPU run.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or
      "No GPU detected. Set Runtime > Change runtime type > T4 GPU and rerun.")

## 3. Install

Clone the canopy height model and install its dependencies, plus the few packages this notebook adds (pyarrow for the sample table, scikit-learn and joblib for the regressor).

**After this cell finishes, restart the runtime** (Runtime > Restart session), then run cell 1 again and continue from cell 4. The install replaces Colab's preinstalled PyTorch, and continuing without a restart leaves a half-loaded CUDA extension that fails at inference time.

In [ ]:
%%bash
set -e
if [ ! -d /content/naip-chm ]; then
  git clone --depth 1 https://github.com/smorf-ntsg/naip-chm.git /content/naip-chm
fi
cd /content/naip-chm
pip install -q -r requirements.txt
pip install -q pyarrow scikit-learn joblib
echo
echo "Installed. Now restart the runtime, rerun cell 1, and continue from cell 4."

## 4. Mount Drive and lay out the cache

Inputs are the NAIP images in `naip_mosaic/` and the canopy height models in `chm_raw/` from the main notebook. Everything this notebook writes goes under `nir_synthesis/`. Nothing is written into `naip_mosaic/` or `chm_raw/`, because the main notebook discovers its inputs by globbing those folders.

Large rasters are copied to local disk before they are read in blocks. Reading windows through the Drive mount is slow.

In [ ]:
import logging
import os
import shutil
import warnings

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
DRIVE_COND = Path("/content/drive/MyDrive/CSDV_Elkinsville/conditioning_data")
DRIVE_NAIP = DRIVE_ROOT / "naip_mosaic"
DRIVE_CHM_RAW = DRIVE_ROOT / "chm_raw"

DRIVE_SYN = DRIVE_ROOT / "nir_synthesis"
DIR_TABLES = DRIVE_SYN / "tables"
DIR_MODELS = DRIVE_SYN / "models"
DIR_SYN_NAIP = DRIVE_SYN / "naip_synnir"
DIR_SYN_CHM = DRIVE_SYN / "chm_synnir"
DIR_FIG = DRIVE_SYN / "figures"
for d in (DRIVE_COND, DIR_TABLES, DIR_MODELS, DIR_SYN_NAIP, DIR_SYN_CHM, DIR_FIG):
    d.mkdir(parents=True, exist_ok=True)

LOCAL = Path("/content/work")
LOCAL_NAIP = LOCAL / "naip"
LOCAL_SYN = LOCAL / "synnir"
LOCAL_CHM = LOCAL / "chm"
for d in (LOCAL_NAIP, LOCAL_SYN, LOCAL_CHM):
    d.mkdir(parents=True, exist_ok=True)

# Point the model's conditioning directory at the Drive copy.
local_cond = NAIPCHM_REPO / "data" / "conditioning_data"
local_cond.parent.mkdir(parents=True, exist_ok=True)
if local_cond.is_symlink():
    local_cond.unlink()
elif local_cond.exists():
    shutil.rmtree(local_cond)
os.symlink(DRIVE_COND, local_cond)

# The 2010 quad carries a harmless libtiff ExtraSamples warning on every open.
logging.getLogger("rasterio._env").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*ExtraSamples.*")

print("Outputs:", DRIVE_SYN)

## 5. Conditioning rasters

The five static layers the model conditions on. Downloaded once, then read from the Drive cache.

In [ ]:
REQUIRED = ["elevation.tif", "climate_pca.tif", "soil_pca.tif", "nlcd.tif", "ecoregion.tif"]

missing = [name for name in REQUIRED if not (DRIVE_COND / name).exists()]
if missing:
    print("Downloading:", missing)
    !cd {NAIPCHM_REPO} && echo n | python scripts/download_conditioning_data.py

still_missing = [name for name in REQUIRED if not (DRIVE_COND / name).exists()]
assert not still_missing, f"Conditioning rasters missing: {still_missing}"
for name in REQUIRED:
    print(f"  {name:16s} {(DRIVE_COND / name).stat().st_size / 1e6:8.1f} MB")

## 6. Discover NAIP images and existing canopy height models

The model reads the acquisition date from the first run of eight digits in the filename, so every input stem is checked for exactly one such run. Resolution and era come from the raster itself, not from the filename code. The real-NIR canopy height models are matched with a strict `*<date>_chm.tif` pattern that excludes the MrSID test product and any synthetic output.

In [ ]:
import re

import pandas as pd
import rasterio

DATE_RE = re.compile(r"\d{8,}")


def date_tag_of(stem: str) -> str:
    """Return the single eight-digit date in a NAIP stem, or raise."""
    runs = DATE_RE.findall(stem)
    if len(runs) != 1 or len(runs[0]) != 8:
        raise ValueError(f"{stem}: expected exactly one 8-digit run, found {runs}")
    return runs[0]


def discover_naip(root: Path) -> dict:
    """Map year to raster metadata for every NAIP GeoTIFF under root."""
    meta = {}
    for path in sorted(root.rglob("*.tif")):
        if "synnir" in path.stem:
            continue
        tag = date_tag_of(path.stem)
        year = int(tag[:4])
        if year in meta:
            raise ValueError(f"Two images for {year}: {meta[year]['path'].name}, {path.name}")
        with rasterio.open(path) as src:
            meta[year] = dict(
                path=path,
                stem=path.stem,
                date_tag=tag,
                count=src.count,
                res=float(src.res[0]),
                crs=src.crs.to_string(),
                width=src.width,
                height=src.height,
                dtype=src.dtypes[0],
                nodata=src.nodata,
                colorinterp=",".join(ci.name for ci in src.colorinterp),
            )
    return meta


def find_real_chm(tag: str):
    hits = sorted(DRIVE_CHM_RAW.glob(f"*{tag}_chm.tif"))
    return hits[0] if hits else None


META = discover_naip(DRIVE_NAIP)
for year, m in META.items():
    m["chm"] = find_real_chm(m["date_tag"])
    m["era"] = "1m" if m["res"] >= 0.9 else "0.6m"

table = pd.DataFrame(
    [
        dict(
            year=y,
            file=m["stem"],
            bands=m["count"],
            res_m=m["res"],
            era=m["era"],
            size=f"{m['width']}x{m['height']}",
            dtype=m["dtype"],
            nodata=m["nodata"],
            colorinterp=m["colorinterp"],
            chm=(m["chm"].name if m["chm"] else None),
        )
        for y, m in sorted(META.items())
    ]
)
display(table)

FOUR_BAND_YEARS = sorted(y for y, m in META.items() if m["count"] == 4)
assert TARGET_YEAR in META, f"No {TARGET_YEAR} image found under {DRIVE_NAIP}"
assert META[TARGET_YEAR]["count"] == 3, f"{TARGET_YEAR} has {META[TARGET_YEAR]['count']} bands"
assert all(META[y]["dtype"] == "uint8" for y in META), "All images must be uint8"
assert len({m["crs"] for m in META.values()}) == 1, "Images use more than one CRS"
missing_chm = [y for y in FOUR_BAND_YEARS if META[y]["chm"] is None]
assert not missing_chm, f"No real-NIR CHM for {missing_chm}; run the main notebook first"
for y in FOUR_BAND_YEARS:
    with rasterio.open(META[y]["chm"]) as chm:
        assert abs(chm.res[0] - 0.6) < 0.01, f"{y}: CHM resolution {chm.res}"
        assert chm.crs.to_string() == META[y]["crs"], f"{y}: CHM CRS differs from image"
ERA = {y: META[y]["era"] for y in META}
print("4-band years:", FOUR_BAND_YEARS)
print("1 m years:", [y for y in FOUR_BAND_YEARS if ERA[y] == "1m"])

## 7. Feature helpers

Two feature sets. `rgb` is the three bands plus derived brightness, chromaticity and greenness indices. `rgb_texture` adds the mean and standard deviation of each band in a 5 by 5 window at native resolution, which is 5 m at 1 m and 3 m at 0.6 m. That mismatch is a known confound when a texture model trained on one era predicts the other.

Both the sampling step and the full-image prediction call `build_features`, so training and inference features cannot drift apart. The valid mask treats pixels with R, G and B all zero as the image collar.

In [ ]:
from scipy import ndimage

BASE_NAMES = ["R", "G", "B"]
RGB_DERIVED_NAMES = [
    "brightness", "chrom_r", "chrom_g", "chrom_b",
    "g_minus_r", "g_minus_b", "r_minus_b", "exg", "vari", "range",
]
TEXTURE_NAMES = [f"tex_{s}_{b}" for s in ("mean", "std") for b in ("r", "g", "b")]


def feature_names(feature_set: str) -> list:
    names = BASE_NAMES + RGB_DERIVED_NAMES
    if feature_set == "rgb_texture":
        names = names + TEXTURE_NAMES
    elif feature_set != "rgb":
        raise ValueError(f"Unknown feature set {feature_set}")
    return names


def valid_mask(bands: np.ndarray, opening: bool = COLLAR_OPENING) -> np.ndarray:
    """True where the pixel is not collar (R, G, B all zero)."""
    valid = ~np.all(bands[:3] == 0, axis=0)
    if opening:
        valid = ndimage.binary_opening(valid, structure=np.ones((5, 5), bool))
    return valid


def rgb_derived(rgb: np.ndarray) -> np.ndarray:
    """Derived per-pixel features from an (n, 3) float32 RGB array."""
    r, g, b = rgb[:, 0], rgb[:, 1], rgb[:, 2]
    total = r + g + b + 1.0
    return np.column_stack(
        [
            (r + g + b) / 3.0,
            r / total,
            g / total,
            b / total,
            g - r,
            g - b,
            r - b,
            2.0 * g - r - b,
            (g - r) / (np.abs(g + r - b) + 1.0),
            rgb.max(axis=1) - rgb.min(axis=1),
        ]
    ).astype(np.float32)


def texture_planes(rgb_u8: np.ndarray, size: int = TEXTURE_WINDOW) -> np.ndarray:
    """(3, h, w) uint8 -> (6, h, w) float32: window means then window stds."""
    out = np.empty((6,) + rgb_u8.shape[1:], np.float32)
    for i in range(3):
        x = rgb_u8[i].astype(np.float32)
        mean = ndimage.uniform_filter(x, size, mode="reflect")
        mean_sq = ndimage.uniform_filter(x * x, size, mode="reflect")
        out[i] = mean
        out[3 + i] = np.sqrt(np.maximum(mean_sq - mean * mean, 0.0))
        del x, mean, mean_sq
    return out


def texture_at(rgb_u8: np.ndarray, rows: np.ndarray, cols: np.ndarray,
               size: int = TEXTURE_WINDOW) -> np.ndarray:
    """Texture features at sample pixels, one band at a time to bound memory."""
    out = np.empty((len(rows), 6), np.float32)
    for i in range(3):
        x = rgb_u8[i].astype(np.float32)
        mean = ndimage.uniform_filter(x, size, mode="reflect")
        out[:, i] = mean[rows, cols]
        mean_sq = ndimage.uniform_filter(x * x, size, mode="reflect")
        out[:, 3 + i] = np.sqrt(np.maximum(mean_sq[rows, cols] - out[:, i] ** 2, 0.0))
        del x, mean, mean_sq
    return out


def build_features(rgb: np.ndarray, feature_set: str, texture=None) -> np.ndarray:
    """(n, 3) RGB (+ (n, 6) texture) -> (n, k) float32 feature matrix."""
    rgb = np.asarray(rgb, np.float32)
    X = np.column_stack([rgb, rgb_derived(rgb)])
    if feature_set == "rgb_texture":
        if texture is None:
            raise ValueError("rgb_texture needs the texture array")
        X = np.column_stack([X, np.asarray(texture, np.float32)])
    elif feature_set != "rgb":
        raise ValueError(f"Unknown feature set {feature_set}")
    return np.ascontiguousarray(X, dtype=np.float32)


def ndvi(nir: np.ndarray, red: np.ndarray) -> np.ndarray:
    nir = nir.astype(np.float32)
    red = red.astype(np.float32)
    return (nir - red) / (nir + red + 1e-6)


def features_from_df(df: pd.DataFrame, feature_set: str) -> np.ndarray:
    rgb = df[BASE_NAMES].to_numpy(np.float32)
    tex = df[TEXTURE_NAMES].to_numpy(np.float32) if feature_set == "rgb_texture" else None
    return build_features(rgb, feature_set, tex)


print("rgb features:", len(feature_names("rgb")), " rgb_texture features:", len(feature_names("rgb_texture")))

## 8. Sampling helpers

Random valid pixels are drawn by rejection so no full index array is built. The canopy height at each sample is read by geographic coordinate: pixel centres from the image transform are converted to row and column on the canopy height model's own 0.6 m grid. The 1 m images sit on a different grid from their canopy height models, so array indices cannot be reused.

In [ ]:
import time

from rasterio.transform import rowcol, xy


def copy_checked(src: Path, dst: Path) -> Path:
    """Copy and confirm the byte size, because the Drive mount can truncate."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    if src.stat().st_size != dst.stat().st_size:
        raise IOError(f"Size mismatch after copy: {src} -> {dst}")
    return dst


def local_copy(path: Path) -> Path:
    dst = LOCAL_NAIP / path.name
    if dst.exists() and dst.stat().st_size == path.stat().st_size:
        return dst
    return copy_checked(path, dst)


def draw_valid_pixels(valid: np.ndarray, n: int, rng: np.random.Generator):
    h, w = valid.shape
    rows = rng.integers(0, h, 3 * n)
    cols = rng.integers(0, w, 3 * n)
    keep = valid[rows, cols]
    rows, cols = rows[keep][:n], cols[keep][:n]
    if len(rows) < n:
        print(f"  warning: only {len(rows)} valid samples drawn (asked {n})")
    return rows, cols


def sample_chm_at(img_src, chm_path: Path, rows: np.ndarray, cols: np.ndarray) -> np.ndarray:
    """Canopy height (cm) at image pixel centres, NaN where nodata or outside."""
    out = np.full(len(rows), np.nan, np.float32)
    if chm_path is None:
        return out
    xs, ys = xy(img_src.transform, rows, cols, offset="center")
    with rasterio.open(chm_path) as chm:
        assert chm.crs == img_src.crs
        r, c = rowcol(chm.transform, xs, ys)
        r, c = np.asarray(r), np.asarray(c)
        inb = (r >= 0) & (r < chm.height) & (c >= 0) & (c < chm.width)
        arr = chm.read(1)
    v = arr[r[inb], c[inb]].astype(np.float32)
    v[v == CHM_NODATA] = np.nan
    out[inb] = v
    return out


def sample_year(year: int, n: int, rng: np.random.Generator) -> pd.DataFrame:
    m = META[year]
    started = time.perf_counter()
    path = local_copy(m["path"])
    with rasterio.open(path) as src:
        bands = src.read()
        valid = valid_mask(bands)
        rows, cols = draw_valid_pixels(valid, n, rng)
        xs, ys = xy(src.transform, rows, cols, offset="center")
        chm_cm = sample_chm_at(src, m["chm"], rows, cols)
    df = pd.DataFrame(
        {
            "year": np.full(len(rows), year, np.int16),
            "era": m["era"],
            "res_m": np.float32(m["res"]),
            "row": rows.astype(np.int32),
            "col": cols.astype(np.int32),
            "x": np.asarray(xs, np.float64),
            "y": np.asarray(ys, np.float64),
            "R": bands[0, rows, cols].astype(np.float32),
            "G": bands[1, rows, cols].astype(np.float32),
            "B": bands[2, rows, cols].astype(np.float32),
            "NIR": (bands[3, rows, cols].astype(np.float32) if bands.shape[0] >= 4
                    else np.full(len(rows), np.nan, np.float32)),
        }
    )
    tex = texture_at(bands[:3], rows, cols)
    for i, name in enumerate(TEXTURE_NAMES):
        df[name] = tex[:, i]
    df["chm_cm"] = chm_cm
    df["collar_frac"] = np.float32(1.0 - valid.mean())
    del bands, valid
    print(f"{year}: {len(df):,} samples, collar {100 * df['collar_frac'].iloc[0]:.1f}%, "
          f"CHM missing {100 * np.isnan(chm_cm).mean():.1f}%, {time.perf_counter() - started:.0f} s")
    return df

## 9. Sample every year once

One parquet on Drive holds the samples for all years, including RGB-only samples from 2005 for the domain-shift diagnostics. Delete the file to resample.

In [ ]:
SAMPLES_PARQUET = DIR_TABLES / f"samples_n{N_SAMPLES}_seed{SEED}_v1.parquet"

if SAMPLES_PARQUET.exists():
    samples = pd.read_parquet(SAMPLES_PARQUET)
    print(f"Loaded {len(samples):,} cached samples from {SAMPLES_PARQUET.name}")
else:
    parts = []
    for year in FOUR_BAND_YEARS + [TARGET_YEAR]:
        parts.append(sample_year(year, N_SAMPLES, np.random.default_rng(SEED + year)))
    samples = pd.concat(parts, ignore_index=True)
    local = LOCAL / SAMPLES_PARQUET.name
    samples.to_parquet(local, index=False)
    copy_checked(local, SAMPLES_PARQUET)
    print(f"Wrote {len(samples):,} samples to {SAMPLES_PARQUET}")

display(samples.groupby("year").agg(n=("R", "size"), era=("era", "first"),
                                    collar_frac=("collar_frac", "first"),
                                    chm_missing=("chm_cm", lambda s: s.isna().mean())))

## 10. Domain shift between years

The 4-band and 3-band images come from different cameras and different eras. Per-year percentiles and histograms of the RGB bands show how far the 2005 image sits from the training years. A 2005 histogram outside the envelope of the training years means the regressor will extrapolate.

In [ ]:
import matplotlib.pyplot as plt

ERA_COLORS = {"1m": "#2a78d6", "0.6m": "#eb6834"}
TARGET_COLOR = "#0b0b0b"
PCTS = [5, 25, 50, 75, 95]

rows = []
for year, g in samples.groupby("year"):
    for band in ["R", "G", "B", "NIR"]:
        v = g[band].dropna().to_numpy()
        if v.size == 0:
            continue
        rows.append(dict(year=year, era=ERA[year], band=band,
                         **{f"p{p}": float(np.percentile(v, p)) for p in PCTS}))
    if g["NIR"].notna().any():
        v = ndvi(g["NIR"].to_numpy(), g["R"].to_numpy())
        rows.append(dict(year=year, era=ERA[year], band="NDVI",
                         **{f"p{p}": float(np.percentile(v, p)) for p in PCTS}))
pct = pd.DataFrame(rows)
pct.to_csv(DIR_TABLES / "domain_shift_percentiles.csv", index=False)
display(pct.pivot(index="year", columns="band", values="p50").round(2))

fig, axes = plt.subplots(4, 1, figsize=(9, 11), sharex=True)
bins = np.arange(0, 257, 2)
for ax, band in zip(axes, ["R", "G", "B", "NIR"]):
    for year, g in samples.groupby("year"):
        v = g[band].dropna().to_numpy()
        if v.size == 0:
            continue
        is_target = year == TARGET_YEAR
        ax.hist(v, bins=bins, density=True, histtype="step",
                color=TARGET_COLOR if is_target else ERA_COLORS[ERA[year]],
                lw=2.2 if is_target else 1.0, alpha=1.0 if is_target else 0.8,
                label=f"{year} ({'3-band' if is_target else ERA[year]})")
    ax.set_ylabel(f"{band} density")
    ax.spines[["top", "right"]].set_visible(False)
axes[-1].set_xlabel("Digital number")
axes[0].legend(ncol=4, fontsize=8, frameon=False, loc="upper right")
axes[0].set_title("Per-year band histograms of sampled pixels (black = 2005 target)")
fig.tight_layout()
fig.savefig(DIR_FIG / "domain_shift.png", dpi=150)
plt.show()

## 11. Regressor and metric helpers

Bias is predicted minus observed. NDVI metrics use the real red band with real and predicted NIR, since the canopy height model presumably reads NIR mostly through its contrast with red. Binned errors stratify by the canopy height at the sample from the real-NIR model.

In [ ]:
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor


def make_model(kind: str = MODEL_KIND):
    if kind == "hgb":
        return HistGradientBoostingRegressor(**HGB_PARAMS)
    if kind == "rf":
        return RandomForestRegressor(**RF_PARAMS)
    raise ValueError(kind)


def fit_model(df: pd.DataFrame, feature_set: str):
    X = features_from_df(df, feature_set)
    y = df["NIR"].to_numpy(np.float32)
    model = make_model()
    model.fit(X, y)
    return model


def predict_chunks(model, X: np.ndarray, chunk: int = PREDICT_CHUNK) -> np.ndarray:
    out = np.empty(len(X), np.float64)
    for i in range(0, len(X), chunk):
        out[i : i + chunk] = model.predict(X[i : i + chunk])
    return out


def regression_metrics(y: np.ndarray, yhat: np.ndarray) -> dict:
    y = y.astype(np.float64)
    yhat = yhat.astype(np.float64)
    d = yhat - y
    ss_res = float((d * d).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return dict(
        n=int(len(y)),
        r2=1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
        rmse=float(np.sqrt((d * d).mean())),
        bias=float(d.mean()),
        mae=float(np.abs(d).mean()),
        pearson_r=float(np.corrcoef(y, yhat)[0, 1]),
    )


def ndvi_metrics(nir: np.ndarray, nir_hat: np.ndarray, red: np.ndarray) -> dict:
    d = ndvi(nir_hat, red) - ndvi(nir, red)
    return dict(ndvi_rmse=float(np.sqrt((d * d).mean())), ndvi_bias=float(d.mean()))


def binned_errors(y, yhat, chm_cm, bins=HEIGHT_BINS_CM, labels=HEIGHT_BIN_LABELS) -> pd.DataFrame:
    d = (yhat - y).astype(np.float64)
    cat = pd.cut(chm_cm, bins=bins, labels=labels, right=False, include_lowest=True)
    frame = pd.DataFrame({"bin": cat, "d": d}).dropna(subset=["bin"])
    grouped = frame.groupby("bin", observed=False)["d"]
    return pd.DataFrame(
        {
            "bin": labels,
            "n": grouped.size().reindex(labels).fillna(0).astype(int).to_numpy(),
            "bias": grouped.mean().reindex(labels).to_numpy(),
            "rmse": np.sqrt((grouped.apply(lambda s: (s * s).mean())).reindex(labels).to_numpy()),
        }
    )


def model_path(feature_set: str, train_set: str, holdout) -> Path:
    return DIR_MODELS / f"{RUN_TAG}_{feature_set}_{train_set}_holdout{holdout or 'none'}.joblib"


def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def has_row(df: pd.DataFrame, **keys) -> bool:
    if df.empty:
        return False
    m = np.ones(len(df), bool)
    for k, v in keys.items():
        m &= (df[k] == v).to_numpy()
    return bool(m.any())

## 12. Leave-one-year-out evaluation of NIR prediction

For each withheld 4-band year, each feature set and each training set, a regressor is fitted on the pooled samples of the other years and evaluated on the withheld year's samples. Rows already in the metrics table are skipped, and each fitted model is saved so the inference loop below reuses it.

In [ ]:
LOO_METRICS_CSV = DIR_TABLES / f"loo_nir_metrics_{RUN_TAG}.csv"
LOO_BINNED_CSV = DIR_TABLES / f"loo_nir_binned_{RUN_TAG}.csv"
loo = load_csv(LOO_METRICS_CSV)
loo_binned = load_csv(LOO_BINNED_CSV)
pool = samples[samples["year"] != TARGET_YEAR]

for year in FOUR_BAND_YEARS:
    test = pool[pool["year"] == year]
    y_test = test["NIR"].to_numpy(np.float32)
    for fs in FEATURE_SETS:
        for ts, ts_years in TRAIN_SETS.items():
            train_years = [y for y in (ts_years or FOUR_BAND_YEARS) if y != year]
            if len(train_years) < 2:
                continue
            if has_row(loo, year=year, feature_set=fs, train_set=ts):
                continue
            train = pool[pool["year"].isin(train_years)]
            mp = model_path(fs, ts, year)
            started = time.perf_counter()
            if mp.exists():
                model = joblib.load(mp)
                fit_s = 0.0
            else:
                model = fit_model(train, fs)
                joblib.dump(model, mp)
                fit_s = time.perf_counter() - started
            yhat = predict_chunks(model, features_from_df(test, fs))
            row = dict(year=year, era=ERA[year], feature_set=fs, train_set=ts,
                       n_train_years=len(train_years), n_train=len(train), fit_s=round(fit_s, 1))
            row.update(regression_metrics(y_test, yhat))
            row.update(ndvi_metrics(y_test, yhat, test["R"].to_numpy()))
            loo = pd.concat([loo, pd.DataFrame([row])], ignore_index=True)
            b = binned_errors(y_test, yhat, test["chm_cm"].to_numpy())
            b.insert(0, "train_set", ts)
            b.insert(0, "feature_set", fs)
            b.insert(0, "era", ERA[year])
            b.insert(0, "year", year)
            loo_binned = pd.concat([loo_binned, b], ignore_index=True)
            loo.to_csv(LOO_METRICS_CSV, index=False)
            loo_binned.to_csv(LOO_BINNED_CSV, index=False)
            print(f"{year} {fs:12s} {ts:6s} r2={row['r2']:.3f} rmse={row['rmse']:.2f} "
                  f"bias={row['bias']:+.2f} ndvi_rmse={row['ndvi_rmse']:.3f} fit {fit_s:.0f} s")

print("\nR2 of predicted NIR on the withheld year")
display(loo.pivot_table(index="year", columns=["feature_set", "train_set"], values="r2").round(3))
print("\nRMSE (digital numbers)")
display(loo.pivot_table(index="year", columns=["feature_set", "train_set"], values="rmse").round(2))
print("\nNDVI RMSE")
display(loo.pivot_table(index="year", columns=["feature_set", "train_set"], values="ndvi_rmse").round(3))

## 13. NIR error by canopy height

NIR error stratified by the canopy height at the sample. Errors concentrated in tall canopy matter more for the canopy height model than errors on bare ground.

In [ ]:
fig, axes = plt.subplots(1, len(FEATURE_SETS), figsize=(5.5 * len(FEATURE_SETS), 4.5), sharey=True)
axes = np.atleast_1d(axes)
sub = loo_binned[loo_binned["train_set"] == "all"]
for ax, fs in zip(axes, FEATURE_SETS):
    for year, g in sub[sub["feature_set"] == fs].groupby("year"):
        g = g.set_index("bin").reindex(HEIGHT_BIN_LABELS)
        ax.plot(range(len(HEIGHT_BIN_LABELS)), g["rmse"], marker="o", ms=4, lw=1.5,
                color=ERA_COLORS[ERA[year]], alpha=0.85, label=str(year))
    ax.set_xticks(range(len(HEIGHT_BIN_LABELS)), HEIGHT_BIN_LABELS)
    ax.set_title(f"Feature set: {fs} (train: all other years)")
    ax.set_xlabel("Canopy height at sample (real-NIR CHM)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("NIR RMSE (digital numbers)")
axes[0].legend(title="Withheld year", fontsize=8, frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(DIR_FIG / f"loo_nir_binned_{RUN_TAG}.png", dpi=150)
plt.show()

## 14. Full-image NIR prediction and the 5-band writer

The image is processed in row blocks. For the `rgb` feature set the prediction runs once per unique RGB triplet in the block and is mapped back, which cuts the work from tens of millions of pixels to a few million. That shortcut is only valid for features that depend on the pixel alone. For `rgb_texture` each block is read with a two-row halo so the window statistics are exact at block edges, and every valid pixel is predicted.

Predicted NIR is rounded and clipped to 0 to 255. Band 5 is the validity mask, which the model uses to blank the collar. Output stems keep the eight-digit date as the only long digit run so the model still reads the correct day of year.

In [ ]:
from rasterio.windows import Window


def synnir_stem(stem: str, feature_set: str, train_set=None) -> str:
    tag = date_tag_of(stem)
    out = f"{stem}_synnir_{feature_set}"
    if train_set:
        out += f"_{train_set}"
    out += f"_{MODEL_KIND}"
    if DATE_RE.findall(out) != [tag]:
        raise ValueError(f"Output stem {out} would confuse the model's date parser")
    return out


def predict_block_rgb(model, rgb_u8: np.ndarray, valid: np.ndarray):
    """Predict NIR for a (3, h, w) block using unique RGB triplets."""
    R, G, B = rgb_u8[0], rgb_u8[1], rgb_u8[2]
    nir = np.zeros(R.shape, np.uint8)
    key = (R.astype(np.uint32) << 16) | (G.astype(np.uint32) << 8) | B.astype(np.uint32)
    kv = key[valid]
    if kv.size == 0:
        return nir, 0
    uniq, inv = np.unique(kv, return_inverse=True)
    trip = np.stack([(uniq >> 16) & 255, (uniq >> 8) & 255, uniq & 255], axis=1).astype(np.float32)
    pred = predict_chunks(model, build_features(trip, "rgb"))[inv]
    n_clip = int(((pred < 0) | (pred > 255)).sum())
    nir[valid] = np.clip(np.rint(pred), 0, 255).astype(np.uint8)
    return nir, n_clip


def predict_block_texture(model, rgb_u8: np.ndarray, valid: np.ndarray):
    """Predict NIR for a (3, h, w) block with texture features, valid pixels only."""
    nir = np.zeros(rgb_u8.shape[1:], np.uint8)
    if not valid.any():
        return nir, 0
    tex = texture_planes(rgb_u8)
    rgb = np.stack([rgb_u8[i][valid] for i in range(3)], axis=1).astype(np.float32)
    texv = np.stack([tex[i][valid] for i in range(6)], axis=1)
    del tex
    pred = predict_chunks(model, build_features(rgb, "rgb_texture", texv))
    n_clip = int(((pred < 0) | (pred > 255)).sum())
    nir[valid] = np.clip(np.rint(pred), 0, 255).astype(np.uint8)
    return nir, n_clip


def write_synnir(src_path: Path, dst_path: Path, model, feature_set: str,
                 block_rows: int = BLOCK_ROWS) -> dict:
    """Write R, G, B, predicted NIR (and mask) as a tiled uint8 GeoTIFF."""
    halo = TEXTURE_WINDOW // 2 if feature_set == "rgb_texture" else 0
    started = time.perf_counter()
    n_valid = 0
    n_clip = 0
    with rasterio.open(src_path) as src:
        H, W = src.height, src.width
        profile = dict(
            driver="GTiff", width=W, height=H, count=5 if WRITE_MASK_BAND else 4,
            dtype="uint8", crs=src.crs, transform=src.transform, nodata=None,
            compress="deflate", predictor=2, tiled=True, blockxsize=512, blockysize=512,
            BIGTIFF="IF_SAFER",
        )
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(dst_path, "w", **profile) as dst:
            for r0 in range(0, H, block_rows):
                r1 = min(r0 + block_rows, H)
                a0, a1 = max(r0 - halo, 0), min(r1 + halo, H)
                block = src.read([1, 2, 3], window=Window(0, a0, W, a1 - a0))
                valid = valid_mask(block)
                if feature_set == "rgb":
                    nir, clipped = predict_block_rgb(model, block, valid)
                else:
                    nir, clipped = predict_block_texture(model, block, valid)
                t0, t1 = r0 - a0, r0 - a0 + (r1 - r0)
                win = Window(0, r0, W, r1 - r0)
                dst.write(block[:, t0:t1], indexes=[1, 2, 3], window=win)
                dst.write(nir[t0:t1], 4, window=win)
                if WRITE_MASK_BAND:
                    dst.write((valid[t0:t1] * 255).astype(np.uint8), 5, window=win)
                n_valid += int(valid[t0:t1].sum())
                n_clip += clipped
    return dict(n_valid=n_valid, n_total=H * W, clip_fraction=n_clip / max(n_valid, 1),
                collar_fraction=1.0 - n_valid / (H * W), predict_s=time.perf_counter() - started)

## 15. Canopy height inference wrapper

Same command as the main notebook. The model writes `<input stem>_chm.tif` into the output directory.

The model's `src/inference_utils.py` has a bug in the path taken by 5-band inputs that need resampling to 0.6 m, which is every 1 m image here. It reads the mask band with an integer band index and a three-element output shape, so rasterio returns a two-dimensional array, and the following `mask_data[0]` keeps only the first row. The mask then has the width of the image instead of its shape and the run fails with "boolean index did not match indexed array along axis 0". `patch_inference_utils` rewrites that one line in the cloned repository so the mask keeps its two dimensions. The patch is idempotent and is applied every time this cell runs.

In [ ]:
UPSTREAM_BUG = "mask = mask_data[0] > 0"
UPSTREAM_FIX = "mask = mask_data.reshape(height, width) > 0"


def patch_inference_utils(repo: Path = NAIPCHM_REPO) -> None:
    """Fix the mask read for 5-band inputs that the model resamples to 0.6 m."""
    path = repo / "src" / "inference_utils.py"
    text = path.read_text()
    if UPSTREAM_FIX in text:
        print("inference_utils.py already patched")
        return
    if UPSTREAM_BUG not in text:
        raise RuntimeError("Expected mask line not found in inference_utils.py; check the upstream code")
    path.write_text(text.replace(UPSTREAM_BUG, UPSTREAM_FIX))
    print("Patched inference_utils.py: mask read for resampled 5-band inputs")


patch_inference_utils()


def run_chm_inference(input_tif: Path, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "scripts/inference.py",
        "--naip-quad", str(input_tif),
        "--output-dir", str(out_dir),
        "--model-checkpoint", MODEL_CHECKPOINT,
        "--config", MODEL_CONFIG,
        "--static-rasters-dir", "data/conditioning_data",
        "--chip-size", str(CHIP_SIZE),
        "--chip-overlap", f"{CHIP_OVERLAP:g}",
    ]
    result = subprocess.run(cmd, cwd=NAIPCHM_REPO, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"Inference failed for {input_tif.name}")
    produced = sorted(out_dir.glob("*_chm.tif"))
    if not produced or not produced[-1].name.startswith(input_tif.stem):
        raise RuntimeError(f"No CHM named after {input_tif.stem} in {out_dir}")
    return produced[-1]

## 16. Canopy height models from synthetic NIR, one per withheld year

Uses `FEATURE_SET_FOR_CHM` and the model trained on all other years. Each fold copies the source image to local disk, writes the synthetic 5-band image, runs inference, and copies the canopy height model to Drive. Folds with a cached canopy height model are skipped. Budget about 7 to 9 minutes per fold.

In [ ]:
CHM_FS = FEATURE_SET_FOR_CHM
RUN_LOG_CSV = DIR_TABLES / f"synnir_run_log_{RUN_TAG}.csv"
run_log = load_csv(RUN_LOG_CSV)
syn_chm = {}

for year in FOUR_BAND_YEARS:
    m = META[year]
    out_stem = synnir_stem(m["stem"], CHM_FS)
    drive_chm = DIR_SYN_CHM / f"{out_stem}_chm.tif"
    if drive_chm.exists():
        syn_chm[year] = drive_chm
        print(f"{year}: cached {drive_chm.name}")
        continue

    started = time.perf_counter()
    mp = model_path(CHM_FS, "all", year)
    if mp.exists():
        model = joblib.load(mp)
    else:
        train = pool[pool["year"] != year]
        model = fit_model(train, CHM_FS)
        joblib.dump(model, mp)

    syn_local = LOCAL_SYN / f"{out_stem}.tif"
    syn_drive = DIR_SYN_NAIP / syn_local.name
    if syn_drive.exists():
        copy_checked(syn_drive, syn_local)
        info = dict(n_valid=np.nan, n_total=np.nan, clip_fraction=np.nan, collar_fraction=np.nan,
                    predict_s=0.0)
        print(f"  reusing synthetic image {syn_drive.name} from Drive")
    else:
        src_local = local_copy(META[year]["path"])
        info = write_synnir(src_local, syn_local, model, CHM_FS)
        if KEEP_SYNNIR_IMAGES:
            copy_checked(syn_local, syn_drive)
    t_inf = time.perf_counter()
    chm_local = run_chm_inference(syn_local, LOCAL_CHM / out_stem)
    info["infer_s"] = time.perf_counter() - t_inf
    copy_checked(chm_local, drive_chm)
    syn_chm[year] = drive_chm

    info.update(year=year, era=ERA[year], feature_set=CHM_FS, train_set="all",
                output=drive_chm.name, total_s=time.perf_counter() - started)
    run_log = pd.concat([run_log, pd.DataFrame([info])], ignore_index=True)
    run_log.to_csv(RUN_LOG_CSV, index=False)
    print(f"{year}: {drive_chm.name} predict {info['predict_s']:.0f} s, infer {info['infer_s']:.0f} s, "
          f"clipped {100 * info['clip_fraction']:.2f}%, collar {100 * info['collar_fraction']:.1f}%")

    syn_local.unlink(missing_ok=True)
    (LOCAL_NAIP / META[year]['path'].name).unlink(missing_ok=True)
    shutil.rmtree(LOCAL_CHM / out_stem, ignore_errors=True)

## 17. Canopy height model comparison helpers

Ported from `scripts/figures/compare_mrsid_uncompressed.py` in the CSDV repository. Statistics are accumulated over row chunks at full resolution, map panels are decimated, and the scatter uses a random sample of valid pixel pairs. `align_to_reference` reprojects one canopy height model onto the other's grid only when the two grids are not already offset by a whole number of pixels, which can happen between the 2005 quad and the 2009 quad.

In [ ]:
from dataclasses import dataclass

from rasterio.warp import Resampling, reproject
from rasterio.windows import from_bounds

CHUNK_ROWS = 1024


@dataclass
class ComparisonStats:
    """Pixel-wise agreement between two co-registered CHMs (units: cm)."""

    n_valid: int
    n_total: int
    mean_a: float
    mean_b: float
    bias: float
    mae: float
    rmse: float
    pearson_r: float

    @property
    def valid_fraction(self) -> float:
        return self.n_valid / self.n_total if self.n_total else float("nan")

    def as_dict(self) -> dict:
        return dict(n_valid=self.n_valid, n_total=self.n_total, valid_fraction=self.valid_fraction,
                    mean_a=self.mean_a, mean_b=self.mean_b, bias=self.bias, mae=self.mae,
                    rmse=self.rmse, pearson_r=self.pearson_r)


def grids_compatible(a, b, tol_px: float = 1e-3) -> bool:
    if a.crs != b.crs or not np.allclose(a.res, b.res, rtol=1e-6):
        return False
    dx = (a.transform.c - b.transform.c) / a.res[0]
    dy = (a.transform.f - b.transform.f) / a.res[1]
    return abs(dx - round(dx)) < tol_px and abs(dy - round(dy)) < tol_px


def align_to_reference(src_path: Path, ref_path: Path, out_dir: Path) -> Path:
    """Return src_path, or a copy reprojected (nearest) onto the reference grid."""
    with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
        if grids_compatible(src, ref):
            return src_path
        out_path = out_dir / f"{src_path.stem}_on_{ref_path.stem}.tif"
        if out_path.exists():
            return out_path
        dst = np.full((ref.height, ref.width), CHM_NODATA, np.uint16)
        reproject(
            source=rasterio.band(src, 1), destination=dst,
            src_transform=src.transform, src_crs=src.crs, src_nodata=CHM_NODATA,
            dst_transform=ref.transform, dst_crs=ref.crs, dst_nodata=CHM_NODATA,
            resampling=Resampling.nearest,
        )
        profile = dict(driver="GTiff", width=ref.width, height=ref.height, count=1, dtype="uint16",
                       crs=ref.crs, transform=ref.transform, nodata=CHM_NODATA,
                       compress="deflate", predictor=2, tiled=True)
        out_dir.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **profile) as out:
            out.write(dst, 1)
        print(f"  aligned {src_path.name} onto the grid of {ref_path.name}")
        return out_path


def intersection_windows(src_a, src_b):
    """Return integer pixel windows covering the geographic overlap of two grids."""
    if src_a.crs != src_b.crs:
        raise ValueError(f"CRS mismatch: {src_a.crs} vs {src_b.crs}")
    if not np.allclose(src_a.res, src_b.res, rtol=1e-6):
        raise ValueError(f"Resolution mismatch: {src_a.res} vs {src_b.res}")
    left = max(src_a.bounds.left, src_b.bounds.left)
    right = min(src_a.bounds.right, src_b.bounds.right)
    bottom = max(src_a.bounds.bottom, src_b.bounds.bottom)
    top = min(src_a.bounds.top, src_b.bounds.top)
    if left >= right or bottom >= top:
        raise ValueError("Rasters do not overlap")
    windows = []
    for src in (src_a, src_b):
        w = from_bounds(left, bottom, right, top, transform=src.transform)
        windows.append(Window(col_off=int(round(w.col_off)), row_off=int(round(w.row_off)),
                              width=int(round(w.width)), height=int(round(w.height))))
    wa, wb = windows
    h = min(wa.height, wb.height)
    w = min(wa.width, wb.width)
    wa = Window(col_off=wa.col_off, row_off=wa.row_off, width=w, height=h)
    wb = Window(col_off=wb.col_off, row_off=wb.row_off, width=w, height=h)
    return wa, wb


def chunked_stats(a: np.ndarray, b: np.ndarray) -> ComparisonStats:
    """Accumulate agreement statistics over row chunks to bound memory."""
    n = 0
    s_x = s_y = s_xx = s_yy = s_xy = s_d = s_ad = s_dd = 0.0
    for r0 in range(0, a.shape[0], CHUNK_ROWS):
        x = a[r0 : r0 + CHUNK_ROWS]
        y = b[r0 : r0 + CHUNK_ROWS]
        m = (x != CHM_NODATA) & (y != CHM_NODATA)
        xf = x[m].astype(np.float64)
        yf = y[m].astype(np.float64)
        d = yf - xf
        n += xf.size
        s_x += xf.sum(); s_y += yf.sum()
        s_xx += (xf * xf).sum(); s_yy += (yf * yf).sum(); s_xy += (xf * yf).sum()
        s_d += d.sum(); s_ad += np.abs(d).sum(); s_dd += (d * d).sum()
    mean_x, mean_y = s_x / n, s_y / n
    cov = s_xy / n - mean_x * mean_y
    var_x = s_xx / n - mean_x**2
    var_y = s_yy / n - mean_y**2
    return ComparisonStats(n_valid=n, n_total=int(a.size), mean_a=mean_x, mean_b=mean_y,
                           bias=s_d / n, mae=s_ad / n, rmse=float(np.sqrt(s_dd / n)),
                           pearson_r=float(cov / np.sqrt(var_x * var_y)))


def binned_chm_stats(a: np.ndarray, b: np.ndarray, bins=HEIGHT_BINS_CM,
                     labels=HEIGHT_BIN_LABELS) -> pd.DataFrame:
    """Bias and RMSE of b relative to a, stratified by the height in a (chunked)."""
    k = len(labels)
    n = np.zeros(k); s_d = np.zeros(k); s_dd = np.zeros(k)
    edges = np.asarray(bins, np.float64)
    for r0 in range(0, a.shape[0], CHUNK_ROWS):
        x = a[r0 : r0 + CHUNK_ROWS]
        y = b[r0 : r0 + CHUNK_ROWS]
        m = (x != CHM_NODATA) & (y != CHM_NODATA)
        xf = x[m].astype(np.float64)
        d = y[m].astype(np.float64) - xf
        idx = np.clip(np.searchsorted(edges, xf, side="right") - 1, 0, k - 1)
        n += np.bincount(idx, minlength=k)
        s_d += np.bincount(idx, weights=d, minlength=k)
        s_dd += np.bincount(idx, weights=d * d, minlength=k)
    with np.errstate(invalid="ignore", divide="ignore"):
        return pd.DataFrame({"bin": labels, "n": n.astype(int), "bias": s_d / n,
                             "rmse": np.sqrt(s_dd / n)})


def sample_valid_pairs(a: np.ndarray, b: np.ndarray, n: int, rng: np.random.Generator):
    rows = rng.integers(0, a.shape[0], size=2 * n)
    cols = rng.integers(0, a.shape[1], size=2 * n)
    x = a[rows, cols]
    y = b[rows, cols]
    m = (x != CHM_NODATA) & (y != CHM_NODATA)
    return x[m][:n].astype(np.float32), y[m][:n].astype(np.float32)


def decimate(arr: np.ndarray, factor: int) -> np.ndarray:
    sub = arr[::factor, ::factor]
    out = sub.astype(np.float32)
    out[sub == CHM_NODATA] = np.nan
    return out


def make_comparison_figure(a, b, stats, x_samp, y_samp, factor, out_path, title_a, title_b,
                           label_a, label_b, dpi=150):
    a_d = decimate(a, factor)
    b_d = decimate(b, factor)
    diff_d = b_d - a_d
    vmax = float(np.nanpercentile(a_d, 99))
    dlim = float(np.nanpercentile(np.abs(diff_d), 98))

    fig, axes = plt.subplots(2, 2, figsize=(13, 12))
    ax_a, ax_b, ax_d, ax_x = axes.ravel()
    im = ax_a.imshow(a_d, cmap="viridis", vmin=0, vmax=vmax, interpolation="nearest")
    ax_a.set_title(title_a)
    fig.colorbar(im, ax=ax_a, fraction=0.046, pad=0.02, label="Height (cm)")
    im = ax_b.imshow(b_d, cmap="viridis", vmin=0, vmax=vmax, interpolation="nearest")
    ax_b.set_title(title_b)
    fig.colorbar(im, ax=ax_b, fraction=0.046, pad=0.02, label="Height (cm)")
    im = ax_d.imshow(diff_d, cmap="RdBu_r", vmin=-dlim, vmax=dlim, interpolation="nearest")
    ax_d.set_title(f"Difference ({label_b.split(' (')[0]} minus {label_a.split(' (')[0]})")
    fig.colorbar(im, ax=ax_d, fraction=0.046, pad=0.02, label="Height difference (cm)")
    for ax in (ax_a, ax_b, ax_d):
        ax.set_axis_off()

    lim = float(max(np.percentile(x_samp, 99.9), np.percentile(y_samp, 99.9), 1.0))
    ax_x.hexbin(x_samp, y_samp, gridsize=80, extent=(0, lim, 0, lim), bins="log",
                cmap="Greys", mincnt=1, linewidths=0.2)
    ax_x.plot([0, lim], [0, lim], color="#c0392b", lw=1, ls="--", label="1:1")
    ax_x.set_xlim(0, lim); ax_x.set_ylim(0, lim); ax_x.set_aspect("equal")
    ax_x.set_xlabel(label_a); ax_x.set_ylabel(label_b)
    ax_x.set_title(f"Pixel-wise comparison (n = {len(x_samp):,} sampled)")
    ax_x.legend(loc="lower right", frameon=False)
    ax_x.text(0.03, 0.97, (f"r = {stats.pearson_r:.3f}\nbias = {stats.bias:+.1f} cm\n"
                           f"MAE = {stats.mae:.1f} cm\nRMSE = {stats.rmse:.1f} cm"),
              transform=ax_x.transAxes, va="top", ha="left", fontsize=10, family="monospace")
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=dpi)
    plt.show()
    plt.close(fig)


def compare_chms(path_a: Path, path_b: Path, fig_path: Path, title_a: str, title_b: str,
                 label_a: str, label_b: str, seed: int = SEED):
    """Compare CHM b against reference a: stats, height-binned stats, figure."""
    path_b = align_to_reference(path_b, path_a, LOCAL_CHM)
    with rasterio.open(path_a) as s1, rasterio.open(path_b) as s2:
        w1, w2 = intersection_windows(s1, s2)
        a = s1.read(1, window=w1)
        b = s2.read(1, window=w2)
    stats = chunked_stats(a, b)
    binned = binned_chm_stats(a, b)
    x_samp, y_samp = sample_valid_pairs(a, b, SCATTER_N, np.random.default_rng(seed))
    make_comparison_figure(a, b, stats, x_samp, y_samp, DECIMATE, fig_path,
                           title_a, title_b, label_a, label_b)
    del a, b
    return stats, binned

## 18. Compare each synthetic-NIR canopy height model with the real-NIR one

The reference is the canopy height model from the real 4-band image. Bias is synthetic minus real. Binned rows stratify by the reference height so that errors in tall canopy are visible separately from errors on open ground.

In [ ]:
CHM_LOO_CSV = DIR_TABLES / f"loo_chm_metrics_{RUN_TAG}_{CHM_FS}.csv"
CHM_LOO_BINNED_CSV = DIR_TABLES / f"loo_chm_binned_{RUN_TAG}_{CHM_FS}.csv"
chm_loo = load_csv(CHM_LOO_CSV)
chm_loo_binned = load_csv(CHM_LOO_BINNED_CSV)

for year in FOUR_BAND_YEARS:
    if year not in syn_chm or has_row(chm_loo, year=year):
        continue
    print(f"{year}: comparing {syn_chm[year].name} with {META[year]['chm'].name}")
    stats, binned = compare_chms(
        META[year]["chm"], syn_chm[year], DIR_FIG / f"chm_compare_{year}_{CHM_FS}_{MODEL_KIND}.png",
        title_a=f"CHM from real NIR ({year})", title_b=f"CHM from synthetic NIR ({year}, {CHM_FS})",
        label_a="Real-NIR CHM (cm)", label_b="Synthetic-NIR CHM (cm)")
    row = dict(year=year, era=ERA[year], feature_set=CHM_FS, train_set="all", **stats.as_dict())
    chm_loo = pd.concat([chm_loo, pd.DataFrame([row])], ignore_index=True)
    binned.insert(0, "era", ERA[year]); binned.insert(0, "year", year)
    chm_loo_binned = pd.concat([chm_loo_binned, binned], ignore_index=True)
    chm_loo.to_csv(CHM_LOO_CSV, index=False)
    chm_loo_binned.to_csv(CHM_LOO_BINNED_CSV, index=False)

print("Synthetic-NIR CHM against real-NIR CHM, per withheld year (cm)")
display(chm_loo[["year", "era", "n_valid", "mean_a", "mean_b", "bias", "mae", "rmse", "pearson_r"]].round(2))
print("\nBias by reference height class (cm)")
display(chm_loo_binned.pivot(index="year", columns="bin", values="bias")[HEIGHT_BIN_LABELS].round(1))
print("\nRMSE by reference height class (cm)")
display(chm_loo_binned.pivot(index="year", columns="bin", values="rmse")[HEIGHT_BIN_LABELS].round(1))

## 19. Summary across years

Top: R² of predicted NIR on each withheld year for both feature sets. Solid lines use every other year for training, dashed lines with hollow markers use the 1 m years only. Bottom: RMSE and bias of the synthetic-NIR canopy height model against the real-NIR one. Shaded years are 1 m imagery.

In [ ]:
FS_COLORS = {"rgb": "#2a78d6", "rgb_texture": "#eb6834"}
INK = "#0b0b0b"

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
for ax in (ax1, ax2):
    for y in FOUR_BAND_YEARS:
        if ERA[y] == "1m":
            ax.axvspan(y - 0.5, y + 0.5, color="#000000", alpha=0.06, lw=0)
    ax.spines[["top", "right"]].set_visible(False)

for fs in FEATURE_SETS:
    for ts, ls, mfc in (("all", "-", FS_COLORS[fs]), ("early", "--", "white")):
        g = loo[(loo["feature_set"] == fs) & (loo["train_set"] == ts)].sort_values("year")
        if g.empty:
            continue
        ax1.plot(g["year"], g["r2"], ls=ls, lw=2, marker="o", ms=7, color=FS_COLORS[fs],
                 markerfacecolor=mfc, markeredgewidth=1.5, label=f"{fs}, train {ts}")
ax1.set_ylabel("R² of predicted NIR (withheld year)")
ax1.legend(frameon=False, fontsize=9, ncol=2)
ax1.set_title("Leave-one-year-out NIR prediction (shaded = 1 m imagery)")

if not chm_loo.empty:
    g = chm_loo.sort_values("year")
    ax2.plot(g["year"], g["rmse"], lw=2, marker="o", ms=7, color=INK, label="RMSE")
    ax2.plot(g["year"], g["bias"], lw=2, ls="--", marker="s", ms=6, color="#52514e", label="Bias (synthetic minus real)")
    ax2.axhline(0, color="#c3c2b7", lw=1)
    ax2.legend(frameon=False, fontsize=9)
ax2.set_ylabel("CHM error (cm)")
ax2.set_xlabel("Withheld year")
ax2.set_title(f"Synthetic-NIR CHM against real-NIR CHM ({CHM_FS}, train all)")
ax2.set_xticks(FOUR_BAND_YEARS)
fig.tight_layout()
fig.savefig(DIR_FIG / f"summary_by_year_{RUN_TAG}_{CHM_FS}.png", dpi=150)
plt.show()

## 20. Synthetic NIR and canopy height for 2005

Two models, each fitted on every sample of its training set: all 4-band years, and the 1 m years only. Both use `FEATURE_SET_FOR_CHM`. The collar of the 2005 image is detected as pixels with R, G and B all zero and written to band 5 so the model blanks it.

In [ ]:
m05 = META[TARGET_YEAR]
chm_2005 = {}
for ts, ts_years in TRAIN_SETS.items():
    out_stem = synnir_stem(m05["stem"], CHM_FS, ts)
    drive_chm = DIR_SYN_CHM / f"{out_stem}_chm.tif"
    if drive_chm.exists():
        chm_2005[ts] = drive_chm
        print(f"{TARGET_YEAR} {ts}: cached {drive_chm.name}")
        continue
    started = time.perf_counter()
    mp = model_path(CHM_FS, ts, None)
    if mp.exists():
        model = joblib.load(mp)
    else:
        train = pool[pool["year"].isin(ts_years or FOUR_BAND_YEARS)]
        model = fit_model(train, CHM_FS)
        joblib.dump(model, mp)
        print(f"  fitted {ts} model on {len(train):,} samples in {time.perf_counter() - started:.0f} s")
    syn_local = LOCAL_SYN / f"{out_stem}.tif"
    syn_drive = DIR_SYN_NAIP / syn_local.name
    if syn_drive.exists():
        copy_checked(syn_drive, syn_local)
        info = dict(n_valid=np.nan, n_total=np.nan, clip_fraction=np.nan, collar_fraction=np.nan,
                    predict_s=0.0)
        print(f"  reusing synthetic image {syn_drive.name} from Drive")
    else:
        src_local = local_copy(m05["path"])
        info = write_synnir(src_local, syn_local, model, CHM_FS)
        if KEEP_SYNNIR_IMAGES:
            copy_checked(syn_local, syn_drive)
    t_inf = time.perf_counter()
    chm_local = run_chm_inference(syn_local, LOCAL_CHM / out_stem)
    info["infer_s"] = time.perf_counter() - t_inf
    copy_checked(chm_local, drive_chm)
    chm_2005[ts] = drive_chm
    info.update(year=TARGET_YEAR, era=ERA[TARGET_YEAR], feature_set=CHM_FS, train_set=ts,
                output=drive_chm.name, total_s=time.perf_counter() - started)
    run_log = pd.concat([run_log, pd.DataFrame([info])], ignore_index=True)
    run_log.to_csv(RUN_LOG_CSV, index=False)
    print(f"{TARGET_YEAR} {ts}: {drive_chm.name} predict {info['predict_s']:.0f} s, infer {info['infer_s']:.0f} s, "
          f"clipped {100 * info['clip_fraction']:.2f}%, collar {100 * info['collar_fraction']:.1f}%")
    syn_local.unlink(missing_ok=True)
    shutil.rmtree(LOCAL_CHM / out_stem, ignore_errors=True)

## 21. 2005 comparisons and quicklook

There is no real-NIR canopy height model for 2005, so the two 2005 products are compared with each other and with the 2009 real-NIR model, the nearest year. Differences against 2009 mix four years of growth and disturbance with the synthetic-NIR error, so they bound the error rather than measure it.

In [ ]:
CHM_2005_CSV = DIR_TABLES / f"chm2005_metrics_{RUN_TAG}_{CHM_FS}.csv"
chm_2005_rows = []
ref_year = min(FOUR_BAND_YEARS)
pairs = [
    (META[ref_year]["chm"], chm_2005["all"], f"real NIR {ref_year}", "2005 (train all)", "2005_all_vs_ref"),
    (META[ref_year]["chm"], chm_2005["early"], f"real NIR {ref_year}", "2005 (train early)", "2005_early_vs_ref"),
    (chm_2005["all"], chm_2005["early"], "2005 (train all)", "2005 (train early)", "2005_all_vs_early"),
]
for path_a, path_b, name_a, name_b, key in pairs:
    stats, binned = compare_chms(
        path_a, path_b, DIR_FIG / f"chm_{key}_{CHM_FS}_{MODEL_KIND}.png",
        title_a=f"CHM from {name_a}", title_b=f"CHM from synthetic NIR, {name_b}",
        label_a=f"CHM {name_a} (cm)", label_b=f"CHM {name_b} (cm)")
    row = dict(comparison=key, reference=name_a, other=name_b, feature_set=CHM_FS, **stats.as_dict())
    for _, b in binned.iterrows():
        row[f"bias_{b['bin']}"] = b["bias"]
        row[f"rmse_{b['bin']}"] = b["rmse"]
    chm_2005_rows.append(row)
chm_2005_df = pd.DataFrame(chm_2005_rows)
chm_2005_df.to_csv(CHM_2005_CSV, index=False)
display(chm_2005_df[["comparison", "n_valid", "mean_a", "mean_b", "bias", "mae", "rmse", "pearson_r"]].round(2))

# Quicklook: RGB, NDVI from synthetic NIR, CHM, all decimated
syn_path = DIR_SYN_NAIP / f"{synnir_stem(m05['stem'], CHM_FS, 'all')}.tif"
if syn_path.exists():
    with rasterio.open(syn_path) as src:
        shape = (src.height // DECIMATE, src.width // DECIMATE)
        rgbn = src.read([1, 2, 3, 4], out_shape=(4,) + shape, resampling=Resampling.nearest).astype(np.float32)
        mask = src.read(5, out_shape=shape, resampling=Resampling.nearest) > 0 if src.count >= 5 else np.ones(shape, bool)
    with rasterio.open(chm_2005["all"]) as src:
        chm_d = decimate(src.read(1), DECIMATE)
    rgb = np.moveaxis(rgbn[:3], 0, -1)
    lo, hi = np.percentile(rgb[mask], [2, 98])
    rgb = np.clip((rgb - lo) / max(hi - lo, 1), 0, 1)
    rgb[~mask] = 1.0
    nd = ndvi(rgbn[3], rgbn[0]); nd[~mask] = np.nan
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(rgb); axes[0].set_title(f"{TARGET_YEAR} RGB")
    im = axes[1].imshow(nd, cmap="YlGn", vmin=-0.2, vmax=0.8); axes[1].set_title("NDVI from synthetic NIR")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.02)
    im = axes[2].imshow(chm_d, cmap="viridis", vmin=0, vmax=np.nanpercentile(chm_d, 99)); axes[2].set_title("CHM (train all)")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.02, label="Height (cm)")
    for ax in axes:
        ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(DIR_FIG / f"quicklook_{TARGET_YEAR}_{CHM_FS}_{MODEL_KIND}.png", dpi=150)
    plt.show()
else:
    print("Synthetic 2005 image not on Drive (KEEP_SYNNIR_IMAGES is False), quicklook skipped.")

## 22. Summary

Writes a markdown summary and a long-form CSV of every metric to `nir_synthesis/tables/`, and lists every product with its size.

In [ ]:
def md_table(df: pd.DataFrame, floatfmt: str = ".3f") -> str:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype.kind == "f":
            df[c] = df[c].map(lambda v: "" if pd.isna(v) else format(v, floatfmt))
    cols = [" / ".join(str(x) for x in c if str(x)) if isinstance(c, tuple) else str(c)
            for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(v) for v in r.to_numpy()) + " |")
    return "\n".join(lines)


parts = [f"# Synthetic NIR summary ({SITE}, run {RUN_TAG})", "",
         f"Target year {TARGET_YEAR}. Feature set for CHM inference: {CHM_FS}. Regressor: {MODEL_KIND}. "
         f"Samples per year: {N_SAMPLES:,}. Training sets: {dict(TRAIN_SETS)}.", "",
         "## NIR prediction on the withheld year (R2)", "",
         md_table(loo.pivot_table(index="year", columns=["feature_set", "train_set"], values="r2").reset_index()),
         "", "## NIR prediction on the withheld year (RMSE, digital numbers)", "",
         md_table(loo.pivot_table(index="year", columns=["feature_set", "train_set"], values="rmse").reset_index(), ".2f"),
         "", "## NDVI RMSE on the withheld year", "",
         md_table(loo.pivot_table(index="year", columns=["feature_set", "train_set"], values="ndvi_rmse").reset_index()),
         "", f"## Synthetic-NIR CHM against real-NIR CHM ({CHM_FS}, train all), cm", ""]
if not chm_loo.empty:
    parts.append(md_table(chm_loo[["year", "era", "n_valid", "bias", "mae", "rmse", "pearson_r"]], ".2f"))
    parts += ["", "### Bias by reference height class (cm)", "",
              md_table(chm_loo_binned.pivot(index="year", columns="bin", values="bias")[HEIGHT_BIN_LABELS].reset_index(), ".1f")]
parts += ["", f"## {TARGET_YEAR} comparisons (cm)", "",
          md_table(chm_2005_df[["comparison", "n_valid", "bias", "mae", "rmse", "pearson_r"]], ".2f"),
          "", "## Domain shift: median digital number per year", "",
          md_table(pct.pivot(index="year", columns="band", values="p50").reset_index(), ".2f")]

summary_md = "\n".join(parts)
(DIR_TABLES / f"summary_{RUN_TAG}_{CHM_FS}.md").write_text(summary_md)

long = pd.concat([
    loo.assign(table="loo_nir"),
    chm_loo.assign(table="loo_chm"),
    chm_2005_df.assign(table="chm_2005"),
], ignore_index=True, sort=False)
long.to_csv(DIR_TABLES / f"summary_{RUN_TAG}_{CHM_FS}.csv", index=False)

paths = sorted(p for p in DRIVE_SYN.rglob("*") if p.is_file())
(DIR_TABLES / "output_paths.txt").write_text(
    "\n".join(f"{p.stat().st_size / 1e6:9.1f} MB  {p.relative_to(DRIVE_ROOT)}" for p in paths))
print(summary_md)
print(f"\n{len(paths)} files under {DRIVE_SYN}")